# 客户端 transport
了解 FastMCP 客户端连接到服务器的不同方式。

FastMCP Client依赖于一个`ClientTransport`对象来处理与 MCP 服务器的连接和通信细节。FastMCP 为常见的连接方法提供了几种内置的传输实现。

虽然Client通常会自动推断出正确的传输（请参阅客户端概述），但您也可以明确实例化传输以获得更好的控制。

> 客户端是轻量级对象，因此您可以根据需要随意创建新的客户端。但是，请注意上下文管理——每次打开客户端上下文 ( async with client:) 时，都会启动一个新的连接或进程。为了获得最佳性能，请在执行多个操作时保持客户端上下文处于打开状态，而不是反复打开和关闭它们。

## 选择transport方式
- 连接到远程/持久服务器：使用StreamableHttpTransport（推荐，HTTP URL 的默认设置）或SSETransport（旧选项）进行基于 Web 的部署。

- 本地开发/测试：用FastMCPTransportFastMCP 服务器的内存中、相同进程测试。

- 运行本地服务器：如果您需要将 MCP 服务器作为打包工具运行，请使用`UvxStdioTransport`（Python/uv）或`NpxStdioTransport`（Node/npm）。


### 网络传输
这些传输连接到通过网络运行的服务器，通常是可通过 URL 访问的长期运行的服务。

### 可流式传输的 HTTP

可流式 HTTP 是基于 Web 部署的推荐传输方式，可通过 HTTP 提供高效的双向通信。

#### 概述 
- 类: `fastmcp.client.transports.StreamableHttpTransport `
- 推断自以 http:// 或 https:// 开头的 URL（自版本 2.3.0 起 HTTP URL 的默认值） 
- 服务器兼容性：适用于以可流 HTTP 模式运行的 FastMCP 服务器

#### 基本用法 
使用 Streamable HTTP 的最简单方法是从 URL 推断传输：

In [ ]:
from fastmcp import Client
import asyncio

# The Client automatically uses StreamableHttpTransport for HTTP URLs
client = Client("https://example.com/mcp")

async def main():
    async with client:
        tools = await client.list_tools()
        print(f"Available tools: {tools}")

asyncio.run(main())

#### 使用标头进行身份验证
对于需要身份验证的服务器：

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# Create transport with authentication headers
transport = StreamableHttpTransport(
    url="https://example.com/mcp",
    headers={"Authorization": "Bearer your-token-here"}
)

client = Client(transport)

### SSE（服务器发送事件）
服务器发送事件 (SSE) 是一种允许服务器通过 HTTP 连接向客户端推送数据的传输方式。虽然 Streamable HTTP 仍然受支持，但现在已成为新的 Web 部署的推荐传输方式。

#### 概述
- 类： fastmcp.client.transports.SSETransport
- 推断依据：自 v2.3.0 起不再自动推断 HTTP URL（必须明确指定）
- 服务器兼容性：sse与在模式下运行的 FastMCP 服务器兼容

#### 基本用法
从 v2.3.0 开始，您必须明确创建一个SSETransport用于 SSE 连接：

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import SSETransport
import asyncio

# Create an SSE transport
transport = SSETransport(url="https://example.com/sse")

# Pass the transport to the client
client = Client(transport)

async def main():
    async with client:
        tools = await client.list_tools()
        print(f"Available tools: {tools}")

asyncio.run(main())

#### 使用标头进行身份验证
SSE 传输还支持自定义标头进行身份验证：

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import SSETransport

# Create SSE transport with authentication headers
transport = SSETransport(
    url="https://example.com/sse",
    headers={"Authorization": "Bearer your-token-here"}
)

client = Client(transport)

何时使用 SSE 和流式 HTTP

在以下情况下使用 Streamable HTTP：

- 设置新部署（推荐默认）
- 您需要双向流媒体
- 您正在连接到以streamable-http模式运行的 FastMCP 服务器


在以下情况下使用 SSE：

- 连接到以sse模式运行的旧式 FastMCP 服务器
- 使用针对服务器发送事件优化的基础设施

## 本地Transports
这些传输管理作为子进程运行的 MCP 服务器，并通过标准输入 (stdin) 和标准输出 (stdout) 与其通信。这是 Claude Desktop 等客户端使用的标准机制。

### Python 标准输入输出系统
- 类： fastmcp.client.transports.PythonStdioTransport
- 推断来源：.py文件路径
- 用例：在子进程中运行基于 Python 的 MCP 服务器脚本

这是在开发期间或与需要启动服务器脚本的工具集成时与本地 FastMCP 服务器交互的最常见方式。

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport

server_script = "my_mcp_server.py" # Path to your server script

# 选择1:基本: Inferred transport
client = Client(server_script)

#选择2:自定义: Explicit transport with custom configuration
transport = PythonStdioTransport(
    script_path=server_script,
    python_cmd="/usr/bin/python3.11", # Optional: specify Python interpreter
    # args=["--some-server-arg"],      # Optional: pass arguments to the script
    # env={"MY_VAR": "value"},         # Optional: set environment variables
)
client = Client(transport)

async def main():
    async with client:
        tools = await client.list_tools()
        print(f"Connected via Python Stdio, found tools: {tools}")

asyncio.run(main())

### Node.js Stdio
- 类： fastmcp.client.transports.NodeStdioTransport
- 推断来源：.js文件路径
- 用例：在子进程中运行基于 Node.js 的 MCP 服务器脚本

与 Python 传输类似，但用于 JavaScript 服务器。

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import NodeStdioTransport

node_server_script = "my_mcp_server.js" # Path to your Node.js server script

# Option 1: Inferred transport
client = Client(node_server_script)

# Option 2: Explicit transport
transport = NodeStdioTransport(
    script_path=node_server_script,
    node_cmd="node" # Optional: specify path to Node executable
)
client = Client(transport)

async def main():
    async with client:
        tools = await client.list_tools()
        print(f"Connected via Node.js Stdio, found tools: {tools}")

asyncio.run(main())

### UVX Stdio（实验性）
- 类： fastmcp.client.transports.UvxStdioTransport
- 推断依据：无法自动推断
- 用例：使用 Python 工具运行 MCP 服务器uvx

这对于执行作为命令行工具或包分发的 MCP 服务器很有用，而无需将它们安装到您的环境中

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import UvxStdioTransport

# Run a hypothetical 'cloud-analyzer-mcp' tool via uvx
transport = UvxStdioTransport(
    tool_name="cloud-analyzer-mcp",
    # from_package="cloud-analyzer-cli", # Optional: specify package if tool name differs
    # with_packages=["boto3", "requests"] # Optional: add dependencies
)
client = Client(transport)

async def main():
    async with client:
        result = await client.call_tool("analyze_bucket", {"name": "my-data"})
        print(f"Analysis result: {result}")

asyncio.run(main())

### NPX Stdio（实验性）
- 类： fastmcp.client.transports.NpxStdioTransport
- 推断依据：无法自动推断
- 用例：使用以下方式运行打包为 NPM 包的 MCP 服务器npx

类似于UvxStdioTransport，但适用于 Node.js 生态系统。

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import NpxStdioTransport

# Run an MCP server from an NPM package
transport = NpxStdioTransport(
    package="mcp-server-package",
    # args=["--port", "stdio"] # Optional: pass arguments to the package
)
client = Client(transport)

async def main():
    async with client:
        result = await client.call_tool("get_npm_data", {})
        print(f"Result: {result}")

asyncio.run(main())

## 内存传输
### FastMCP 传输
- 类： fastmcp.client.transports.FastMCPTransport
- 推断自：一个实例fastmcp.server.FastMCP
- 用例：FastMCP在同一个 Python 进程中直接连接到服务器实例

这对于测试您的 FastMCP 服务器非常有用。

In [ ]:
from fastmcp import FastMCP, Client
import asyncio

# 1. Create your FastMCP server instance
server = FastMCP(name="InMemoryServer")

@server.tool()
def ping(): 
    return "pong"

# 2. Create a client pointing directly to the server instance
client = Client(server)  # Transport is automatically inferred

async def main():
    async with client:
        result = await client.call_tool("ping")
        print(f"In-memory call result: {result}")

asyncio.run(main())

通信通过高效的内存队列进行，使其非常快并且非常适合单元测试。